# Stage 2 V4: O-LoRA + Data Mixing 70/30

## Résumé des échecs précédents

| Version | Accuracy | Problème |
|---------|----------|----------|
| Stage 1 | **80.52%** | ✅ Classification OK |
| V1 | 45% | Catastrophic forgetting (CMD 95%, autres ~0%) |
| V2 | 31% | Base model, pas Stage 1 |
| V3 | 1% | Perte vocabulaire (réponses génériques françaises) |

## Cause V3
- `is_trainable=True` → Stage 1 LoRA modifié
- Data mix: 97% Stage 2 / 3% Stage 1 (mauvais ratio)

## Solution V4: O-LoRA + Data Mixing 70/30

Basé sur recherche scientifique (WACV 2025, arXiv):
1. **O-LoRA (Orthogonal LoRA)**: Geler Stage 1 LoRA, ajouter nouveau LoRA sur modules différents
2. **Data Mixing 70/30**: 70% Stage 1 + 30% Stage 2 (pas l'inverse!)

### Configuration V4

| Paramètre | V3 (échec) | V4 |
|-----------|------------|-----|
| Stage 1 LoRA | trainable | **FROZEN** |
| Nouveau LoRA | non | **r=8, k_proj+o_proj** |
| Data mix | 97% S2 / 3% S1 | **30% S2 / 70% S1** |
| Epochs | 1 | **2** |

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel
from peft import PeftModel, LoraConfig, get_peft_model

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


In [2]:
# ============================================================
# CELL 2: CONFIGURATION V4
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
STAGE1_CHECKPOINT = BASE_DIR / "checkpoints" / "stage1_cassava_qwen25vl_7b" / "final_model"
STAGE1_DATA_PATH = BASE_DIR / "data" / "stage1" / "cassava_train_balanced.json"
STAGE2_DATA_PATH = BASE_DIR / "data" / "stage2" / "stage2_vqa_dataset.json"
VAL_DATA_PATH = BASE_DIR / "data" / "stage1" / "cassava_val.json"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage2_v4_olora"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle de base
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# ============================================================
# CONFIGURATION V4 - O-LoRA + DATA MIXING 70/30
# ============================================================

# Nouveau LoRA pour Stage 2 (modules DIFFÉRENTS de Stage 1)
# Stage 1 utilisait: q_proj, v_proj
# Stage 2 utilise: k_proj, o_proj (orthogonal)
NEW_LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 16,
    "target_modules": ["k_proj", "o_proj"],  # Différent de Stage 1!
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

# Data mixing: 70% Stage 1 + 30% Stage 2
DATA_MIX_CONFIG = {
    "stage1_ratio": 0.70,  # 70% données Stage 1 (classification)
    "stage2_ratio": 0.30,  # 30% données Stage 2 (VQA)
    "n_stage1": 20000,     # ~70%
    "n_stage2": 8500,      # ~30%
}

# Training Configuration
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,      # Effective batch = 16
    "num_train_epochs": 2,                 # 2 epochs
    "learning_rate": 2e-5,                 # Plus élevé car nouveau LoRA
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "logging_steps": 50,
    "save_steps": 500,
    "eval_steps": 250,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

# Classes
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
CLASS_FULL_NAMES = {
    "CBB": "Cassava Bacterial Blight (CBB)",
    "CBSD": "Cassava Brown Streak Disease (CBSD)",
    "CGM": "Cassava Green Mottle (CGM)",
    "CMD": "Cassava Mosaic Disease (CMD)",
    "Healthy": "Healthy"
}

# Stage 1 results for comparison
STAGE1_RESULTS = {
    "global": 80.52,
    "CBB": 55.13,
    "CBSD": 83.99,
    "CGM": 56.47,
    "CMD": 85.98,
    "Healthy": 75.37
}

print(f"{'='*60}")
print(f"CONFIGURATION STAGE 2 V4 - O-LoRA + DATA MIXING 70/30")
print(f"{'='*60}")
print(f"\n📊 Différences clés vs V3:")
print(f"  - Stage 1 LoRA: FROZEN (vs trainable)")
print(f"  - Nouveau LoRA: r={NEW_LORA_CONFIG['r']}, modules={NEW_LORA_CONFIG['target_modules']}")
print(f"  - Data mix: {DATA_MIX_CONFIG['stage1_ratio']*100:.0f}% S1 / {DATA_MIX_CONFIG['stage2_ratio']*100:.0f}% S2 (vs 3% S1 / 97% S2)")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (nouveau LoRA)")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"{'='*60}")

CONFIGURATION STAGE 2 V4 - O-LoRA + DATA MIXING 70/30

📊 Différences clés vs V3:
  - Stage 1 LoRA: FROZEN (vs trainable)
  - Nouveau LoRA: r=8, modules=['k_proj', 'o_proj']
  - Data mix: 70% S1 / 30% S2 (vs 3% S1 / 97% S2)
  - LR: 2e-05 (nouveau LoRA)
  - Epochs: 2


In [3]:
# ============================================================
# CELL 3: CHARGER MODÈLE BASE + STAGE 1 LORA (FROZEN)
# ============================================================

print("\n" + "="*60)
print("LOADING BASE MODEL + STAGE 1 LORA (FROZEN)")
print("="*60)

print(f"\n1. Loading base model...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print(f"\n2. Loading Stage 1 LoRA weights (FROZEN)...")
print(f"   Checkpoint: {STAGE1_CHECKPOINT}")

# CRITICAL: is_trainable=False pour garder Stage 1 FROZEN
model = PeftModel.from_pretrained(
    model, 
    str(STAGE1_CHECKPOINT),
    is_trainable=False,  # STAGE 1 RESTE GELÉ
    adapter_name="stage1_frozen"
)

print(f"\n3. Loading processor...")
processor = AutoProcessor.from_pretrained(str(STAGE1_CHECKPOINT))

# Vérifier que Stage 1 est bien frozen
stage1_trainable = sum(p.numel() for n, p in model.named_parameters() 
                       if p.requires_grad and "stage1" in n)

print(f"\n✅ Stage 1 LoRA chargé")
print(f"   - Stage 1 accuracy: {STAGE1_RESULTS['global']:.2f}%")
print(f"   - Stage 1 trainable params: {stage1_trainable:,} (devrait être 0)")
print(f"   - Mode: is_trainable=False (FROZEN)")
print("="*60)


LOADING BASE MODEL + STAGE 1 LORA (FROZEN)

1. Loading base model...


==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

2. Loading Stage 1 LoRA weights (FROZEN)...
   Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model

3. Loading processor...

✅ Stage 1 LoRA chargé
   - Stage 1 accuracy: 80.52%
   - Stage 1 trainable params: 0 (devrait être 0)
   - Mode: is_trainable=False (FROZEN)


In [4]:
# ============================================================
# CELL 4: AJOUTER NOUVEAU LORA ORTHOGONAL (TRAINABLE)
# ============================================================

print("\n" + "="*60)
print("ADDING NEW ORTHOGONAL LORA FOR STAGE 2")
print("="*60)

# Configuration du nouveau LoRA
# Modules DIFFÉRENTS de Stage 1 (q_proj, v_proj)
# → k_proj, o_proj pour orthogonalité
new_lora_config = LoraConfig(
    r=NEW_LORA_CONFIG["r"],
    lora_alpha=NEW_LORA_CONFIG["lora_alpha"],
    target_modules=NEW_LORA_CONFIG["target_modules"],
    lora_dropout=NEW_LORA_CONFIG["lora_dropout"],
    bias=NEW_LORA_CONFIG["bias"],
    task_type=NEW_LORA_CONFIG["task_type"],
)

print(f"\n📊 Nouveau LoRA configuration:")
print(f"   - r: {NEW_LORA_CONFIG['r']}")
print(f"   - alpha: {NEW_LORA_CONFIG['lora_alpha']}")
print(f"   - target_modules: {NEW_LORA_CONFIG['target_modules']}")
print(f"   - Stage 1 modules (frozen): ['q_proj', 'v_proj']")

# Ajouter le nouveau adapter
# CORRECTION: adapter_name est le premier argument positional
model.add_adapter("stage2_vqa", new_lora_config)

# Activer UNIQUEMENT le nouveau adapter pour training
model.set_adapter("stage2_vqa")

# Vérifier les paramètres trainables
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\n✅ Nouveau LoRA ajouté")
print(f"   - Adapter actif: stage2_vqa")
print(f"   - Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.4f}%)")
print(f"   - Total params: {total_params:,}")

# Afficher tous les adapters
print(f"\n📋 Adapters disponibles:")
for name in model.peft_config.keys():
    config = model.peft_config[name]
    print(f"   - {name}: modules={config.target_modules}")

print("="*60)


ADDING NEW ORTHOGONAL LORA FOR STAGE 2

📊 Nouveau LoRA configuration:
   - r: 8
   - alpha: 16
   - target_modules: ['k_proj', 'o_proj']
   - Stage 1 modules (frozen): ['q_proj', 'v_proj']

✅ Nouveau LoRA ajouté
   - Adapter actif: stage2_vqa
   - Trainable params: 2,523,136 (0.0501%)
   - Total params: 5,034,568,704

📋 Adapters disponibles:
   - stage1_frozen: modules={'q_proj', 'v_proj'}
   - stage2_vqa: modules={'k_proj', 'o_proj'}


In [5]:
# ============================================================
# CELL 5: PRÉPARER DONNÉES STAGE 1 (70%)
# ============================================================

print("\n" + "="*60)
print("LOADING STAGE 1 DATA (70% of mix)")
print("="*60)

# Charger Stage 1
with open(STAGE1_DATA_PATH, 'r', encoding='utf-8') as f:
    stage1_data_raw = json.load(f)

print(f"Stage 1 raw data: {len(stage1_data_raw)} samples")

def get_class_from_response(response: str) -> str:
    """Extrait la classe de la réponse Stage 1."""
    response_lower = response.lower()
    if "cbb" in response_lower or "bacterial blight" in response_lower:
        return "CBB"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cgm" in response_lower or "green mottle" in response_lower or "green mite" in response_lower:
        return "CGM"
    elif "cmd" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "healthy" in response_lower:
        return "Healthy"
    return "Unknown"

# Ajouter metadata de classe pour chaque sample
stage1_data = []
for s in stage1_data_raw:
    # Extraire la classe depuis la réponse
    cls = "Unknown"
    for turn in s['conversations']:
        if turn['from'] == 'assistant':
            cls = get_class_from_response(turn['value'])
            break
    
    if cls != "Unknown":
        stage1_data.append({
            'id': f"s1_{s['id']}",
            'conversations': s['conversations'],
            'metadata': {
                'source': 'stage1_classification',
                'class': cls,
                'severity': 'general',
                'qa_type': 'identification'
            }
        })

print(f"Stage 1 valid samples: {len(stage1_data)}")

# Distribution Stage 1
stage1_dist = Counter(s['metadata']['class'] for s in stage1_data)
print("\n📊 Distribution Stage 1:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {stage1_dist[cls]} ({stage1_dist[cls]/len(stage1_data)*100:.1f}%)")

# Échantillonner équilibré (n_stage1 samples)
def balanced_sample(data, target_total, class_key='metadata'):
    """Échantillonne de manière équilibrée."""
    by_class = defaultdict(list)
    for s in data:
        cls = s[class_key]['class'] if isinstance(class_key, str) else s['metadata']['class']
        by_class[cls].append(s)
    
    per_class = target_total // len(CLASS_NAMES)
    sampled = []
    
    for cls in CLASS_NAMES:
        cls_samples = by_class[cls]
        if len(cls_samples) >= per_class:
            sampled.extend(random.sample(cls_samples, per_class))
        else:
            # Oversample
            multiplier = (per_class // len(cls_samples)) + 1
            oversampled = (cls_samples * multiplier)[:per_class]
            sampled.extend(oversampled)
    
    random.shuffle(sampled)
    return sampled

stage1_sampled = balanced_sample(stage1_data, DATA_MIX_CONFIG['n_stage1'])
print(f"\n✅ Stage 1 échantillonné: {len(stage1_sampled)} samples (70%)")

# Vérifier distribution finale
stage1_final_dist = Counter(s['metadata']['class'] for s in stage1_sampled)
print("\n📊 Distribution Stage 1 final:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {stage1_final_dist[cls]}")


LOADING STAGE 1 DATA (70% of mix)
Stage 1 raw data: 29276 samples
Stage 1 valid samples: 29276

📊 Distribution Stage 1:
  CBB: 3717 (12.7%)
  CBSD: 5296 (18.1%)
  CGM: 5054 (17.3%)
  CMD: 12000 (41.0%)
  Healthy: 3209 (11.0%)

✅ Stage 1 échantillonné: 20000 samples (70%)

📊 Distribution Stage 1 final:
  CBB: 4000
  CBSD: 4000
  CGM: 4000
  CMD: 4000
  Healthy: 4000


In [6]:
# ============================================================
# CELL 6: CHARGER DONNÉES STAGE 2 (30%)
# ============================================================

print("\n" + "="*60)
print("LOADING STAGE 2 DATA (30% of mix)")
print("="*60)

# Charger Stage 2
with open(STAGE2_DATA_PATH, 'r', encoding='utf-8') as f:
    stage2_data_raw = json.load(f)

print(f"Stage 2 raw data: {len(stage2_data_raw)} samples")

# Distribution Stage 2
stage2_dist = Counter(s['metadata']['class'] for s in stage2_data_raw)
print("\n📊 Distribution Stage 2:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {stage2_dist[cls]} ({stage2_dist[cls]/len(stage2_data_raw)*100:.1f}%)")

# Échantillonner équilibré (n_stage2 samples)
stage2_sampled = balanced_sample(stage2_data_raw, DATA_MIX_CONFIG['n_stage2'])
print(f"\n✅ Stage 2 échantillonné: {len(stage2_sampled)} samples (30%)")

# Vérifier distribution finale
stage2_final_dist = Counter(s['metadata']['class'] for s in stage2_sampled)
print("\n📊 Distribution Stage 2 final:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {stage2_final_dist[cls]}")


LOADING STAGE 2 DATA (30% of mix)
Stage 2 raw data: 29276 samples

📊 Distribution Stage 2:
  CBB: 3717 (12.7%)
  CBSD: 5296 (18.1%)
  CGM: 5054 (17.3%)
  CMD: 12000 (41.0%)
  Healthy: 3209 (11.0%)

✅ Stage 2 échantillonné: 8500 samples (30%)

📊 Distribution Stage 2 final:
  CBB: 1700
  CBSD: 1700
  CGM: 1700
  CMD: 1700
  Healthy: 1700


In [7]:
# ============================================================
# CELL 7: DATA MIX 70/30 (STAGE 1 + STAGE 2)
# ============================================================

print("\n" + "="*60)
print("MIXING DATASETS: 70% STAGE 1 + 30% STAGE 2")
print("="*60)

# Combiner les datasets
mixed_data = stage1_sampled + stage2_sampled
random.seed(42)
random.shuffle(mixed_data)

# Calculer ratios réels
stage1_count = len([s for s in mixed_data if s['metadata']['source'] == 'stage1_classification'])
stage2_count = len(mixed_data) - stage1_count
stage1_pct = stage1_count / len(mixed_data) * 100
stage2_pct = stage2_count / len(mixed_data) * 100

print(f"\n📊 Dataset mixé:")
print(f"  - Stage 1 (classification): {stage1_count} ({stage1_pct:.1f}%)")
print(f"  - Stage 2 (VQA): {stage2_count} ({stage2_pct:.1f}%)")
print(f"  - Total: {len(mixed_data)}")

# Distribution finale par classe
final_dist = Counter(s['metadata']['class'] for s in mixed_data)
print("\n📊 Distribution finale par classe:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {final_dist[cls]} ({final_dist[cls]/len(mixed_data)*100:.1f}%)")

# Vérifier quelques exemples
print("\n📋 Exemples du dataset mixé:")
for i, sample in enumerate(mixed_data[:3]):
    source = sample['metadata']['source']
    cls = sample['metadata']['class']
    q_type = sample['metadata'].get('qa_type', 'N/A')
    print(f"  [{i+1}] Source: {source}, Class: {cls}, Type: {q_type}")

print("\n✅ Data mixing 70/30 terminé!")
print(f"   → C'est l'INVERSE de V3 qui avait 97% Stage 2 / 3% Stage 1")
print("="*60)


MIXING DATASETS: 70% STAGE 1 + 30% STAGE 2

📊 Dataset mixé:
  - Stage 1 (classification): 20000 (70.2%)
  - Stage 2 (VQA): 8500 (29.8%)
  - Total: 28500

📊 Distribution finale par classe:
  CBB: 5700 (20.0%)
  CBSD: 5700 (20.0%)
  CGM: 5700 (20.0%)
  CMD: 5700 (20.0%)
  Healthy: 5700 (20.0%)

📋 Exemples du dataset mixé:
  [1] Source: stage1_classification, Class: CGM, Type: identification
  [2] Source: stage1_classification, Class: CMD, Type: identification
  [3] Source: kaggle, Class: Healthy, Type: severite

✅ Data mixing 70/30 terminé!
   → C'est l'INVERSE de V3 qui avait 97% Stage 2 / 3% Stage 1


In [8]:
# ============================================================
# CELL 8: PRÉPARATION DATASET ET DATA COLLATOR
# ============================================================

def format_sample_for_training(sample):
    """Formate un sample pour l'entraînement."""
    conversations = sample['conversations']
    
    image_path = None
    user_text = ""
    assistant_text = ""
    
    for turn in conversations:
        if turn['from'] == 'user':
            match = re.search(r'<img>(.*?)</img>', turn['value'])
            if match:
                image_path = match.group(1)
            user_text = re.sub(r'Picture \d+: <img>.*?</img>\n?', '', turn['value']).strip()
        elif turn['from'] == 'assistant':
            assistant_text = turn['value']
    
    return {
        "conversations": [
            {"role": "user", "content": f"Picture 1: <image>\n{user_text}"},
            {"role": "assistant", "content": assistant_text}
        ],
        "image_path": image_path,
        "id": sample['id'],
        "class_name": sample['metadata']['class'],
        "source": sample['metadata'].get('source', 'unknown')
    }

print("\nFormatting data...")
all_formatted = [format_sample_for_training(s) for s in tqdm(mixed_data, desc="Formatting")]

# Filtrer images valides
valid_samples = [s for s in all_formatted if s['image_path'] and os.path.exists(s['image_path'])]
print(f"\n✅ Valid samples: {len(valid_samples)} / {len(all_formatted)}")

# Split train/val (95/5)
random.seed(42)
random.shuffle(valid_samples)
split_idx = int(len(valid_samples) * 0.95)
train_formatted = valid_samples[:split_idx]
val_formatted = valid_samples[split_idx:]

print(f"Train: {len(train_formatted)}")
print(f"Val: {len(val_formatted)}")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)


Formatting data...


Formatting: 100%|██████████| 28500/28500 [00:00<00:00, 120211.93it/s]


✅ Valid samples: 28500 / 28500
Train: 27075
Val: 1425


In [9]:
# ============================================================
# CELL 9: DATA COLLATOR
# ============================================================

@dataclass
class QwenDataCollator:
    """Data collator pour Qwen2.5-VL."""
    processor: Any
    max_length: int = 1536

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert('RGB')
                    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                except:
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            else:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image])

            conversations = feature['conversations']
            messages = []
            
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and len(messages) == 0:
                    text_content = content.replace('<image>', '').replace('Picture 1:', '').strip()
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": text_content}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        )

        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=self.max_length
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-20] = -100

        batch["labels"] = labels
        return batch

data_collator = QwenDataCollator(processor=processor)
print("✅ Data collator créé")

✅ Data collator créé


In [10]:
# ============================================================
# CELL 10: TRAINER V4
# ============================================================

class Stage2V4Trainer(Trainer):
    """Trainer pour Stage 2 V4 avec O-LoRA."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("\n" + "="*60)
print("CONFIGURING TRAINER V4")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=3,
    
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    
    optim="adamw_8bit",
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

trainer = Stage2V4Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch * TRAINING_CONFIG['num_train_epochs']

print(f"\n📊 Configuration V4:")
print(f"  - Stage 1 LoRA: FROZEN (préservé)")
print(f"  - Nouveau LoRA: {NEW_LORA_CONFIG['target_modules']}")
print(f"  - Data mix: {stage1_pct:.0f}% Stage 1 / {stage2_pct:.0f}% Stage 2")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']}")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print("="*60)


CONFIGURING TRAINER V4

📊 Configuration V4:
  - Stage 1 LoRA: FROZEN (préservé)
  - Nouveau LoRA: ['k_proj', 'o_proj']
  - Data mix: 70% Stage 1 / 30% Stage 2
  - Train samples: 27,075
  - Effective batch: 16
  - Total steps: ~3,384
  - LR: 2e-05
  - Epochs: 2


In [11]:
# ============================================================
# CELL 11: TRAINING V4
# ============================================================

print("\n" + "="*60)
print("STARTING STAGE 2 V4 TRAINING (O-LoRA)")
print("="*60)
print(f"\n🎯 Stratégie V4 (vs V3 qui a échoué à 1%):")
print(f"  1. Stage 1 LoRA: FROZEN (vs trainable)")
print(f"  2. Nouveau LoRA: k_proj, o_proj (orthogonal à Stage 1)")
print(f"  3. Data mix: {stage1_pct:.0f}% Stage 1 / {stage2_pct:.0f}% Stage 2 (vs 3% / 97%)")
print(f"\n🎯 Objectif: ≥75% global avec capacités VQA")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("STAGE 2 V4 TRAINING COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")


STARTING STAGE 2 V4 TRAINING (O-LoRA)

🎯 Stratégie V4 (vs V3 qui a échoué à 1%):
  1. Stage 1 LoRA: FROZEN (vs trainable)
  2. Nouveau LoRA: k_proj, o_proj (orthogonal à Stage 1)
  3. Data mix: 70% Stage 1 / 30% Stage 2 (vs 3% / 97%)

🎯 Objectif: ≥75% global avec capacités VQA



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 27,075 | Num Epochs = 2 | Total steps = 3,386
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 2,523,136 of 8,297,212,928 (0.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
250,19.137500,2.101219
500,6.878500,0.876322
750,5.900200,0.665298
1000,4.591600,0.568108
1250,4.542200,0.517902
1500,3.924800,0.489314
1750,3.803800,0.476357
2000,3.767000,0.466440
2250,3.630700,0.460022
2500,3.612100,0.456041


Could not locate the best model at /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v4_olora/checkpoint-3000/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.



STAGE 2 V4 TRAINING COMPLETED
Final loss: 6.1363
Total steps: 3386


In [13]:
# ============================================================
# CELL 12: ÉVALUATION CLASSIFICATION
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION V4 - CLASSIFICATION")
print("="*60)

# Pour l'inférence avec deux adapters, on utilise add_weighted_adapter()
# pour créer un adapter combiné
print("\n🔀 Combinaison des adapters Stage 1 + Stage 2...")

try:
    # Méthode 1: Combiner les adapters avec add_weighted_adapter
    model.add_weighted_adapter(
        adapters=["stage1_frozen", "stage2_vqa"],
        weights=[1.0, 1.0],
        adapter_name="combined_v4",
        combination_type="linear"
    )
    model.set_adapter("combined_v4")
    print("✅ Adapters combinés avec add_weighted_adapter()")
except Exception as e:
    print(f"⚠️ add_weighted_adapter échoué: {e}")
    # Méthode 2: Utiliser seulement stage2_vqa (inclut stage1_frozen car frozen)
    # Puisque stage1_frozen était chargé en premier et est frozen,
    # ses poids sont déjà dans le modèle de base
    model.set_adapter("stage2_vqa")
    print("✅ Utilisation de stage2_vqa (stage1 est frozen dans le modèle)")

FastVisionModel.for_inference(model)

def generate_response(image_path: str, question: str, max_new_tokens: int = 256):
    """Génère une réponse."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Erreur"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    return response

def extract_disease(response: str) -> str:
    """Extrait la maladie de la réponse."""
    response_lower = response.lower()
    if "cmd" in response_lower or "mosaïque" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "striure" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bactériose" in response_lower or "bacterial" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "acarien" in response_lower or "green" in response_lower or "mite" in response_lower:
        return "CGM"
    elif "sain" in response_lower or "healthy" in response_lower:
        return "Healthy"
    return "Unknown"

# Questions d'identification (comme Stage 1)
identification_questions = [
    "What cassava disease is shown in this image?",
    "What disease affects this cassava plant?",
    "Quelle maladie affecte cette feuille de manioc ?"
]

print("\n📊 Évaluation sur 200 samples...")
eval_samples = random.sample(val_formatted, min(200, len(val_formatted)))

results = []
for sample in tqdm(eval_samples, desc="Evaluating"):
    # Utiliser question en anglais (comme Stage 1)
    question = random.choice(identification_questions[:2])  # Questions en anglais
    response = generate_response(sample['image_path'], question)
    predicted = extract_disease(response)
    actual = sample['class_name']
    results.append({
        'actual': actual,
        'predicted': predicted,
        'correct': predicted == actual,
        'response': response[:200]
    })

correct = sum(1 for r in results if r['correct'])
total = len(results)
global_accuracy = correct / total * 100

print(f"\n" + "="*60)
print(f"RÉSULTATS V4 vs Stage 1 vs V3")
print(f"="*60)
print(f"\n📊 Accuracy globale:")
print(f"  - Stage 1: {STAGE1_RESULTS['global']:.1f}%")
print(f"  - Stage 2 V3: 1.0% (échec)")
print(f"  - Stage 2 V4: {global_accuracy:.1f}%")

# Par classe
print("\n📈 Par classe (V4 vs Stage 1):")
class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        class_metrics[cls] = acc
        stage1_acc = STAGE1_RESULTS.get(cls, 0)
        diff = acc - stage1_acc
        status = "✅" if diff >= -10 else "⚠️" if diff >= -20 else "❌"
        print(f"  {status} {CLASS_FULL_NAMES[cls]}: {acc:.1f}% (Stage 1: {stage1_acc:.1f}%, Δ{diff:+.1f}%)")


ÉVALUATION V4 - CLASSIFICATION

🔀 Combinaison des adapters Stage 1 + Stage 2...
✅ Adapters combinés avec add_weighted_adapter()

📊 Évaluation sur 200 samples...


Evaluating: 100%|██████████| 200/200 [05:51<00:00,  1.76s/it]


RÉSULTATS V4 vs Stage 1 vs V3

📊 Accuracy globale:
  - Stage 1: 80.5%
  - Stage 2 V3: 1.0% (échec)
  - Stage 2 V4: 38.5%

📈 Par classe (V4 vs Stage 1):
  ✅ Cassava Bacterial Blight (CBB): 70.2% (Stage 1: 55.1%, Δ+15.1%)
  ❌ Cassava Brown Streak Disease (CBSD): 8.3% (Stage 1: 84.0%, Δ-75.7%)
  ✅ Cassava Green Mottle (CGM): 60.0% (Stage 1: 56.5%, Δ+3.5%)
  ❌ Cassava Mosaic Disease (CMD): 50.0% (Stage 1: 86.0%, Δ-36.0%)
  ❌ Healthy: 0.0% (Stage 1: 75.4%, Δ-75.4%)


In [14]:
# ============================================================
# CELL 13: DIAGNOSTIC - VOIR LES RÉPONSES BRUTES
# ============================================================

print("="*60)
print("DIAGNOSTIC: RÉPONSES BRUTES DU MODÈLE")
print("="*60)

# Prendre 10 samples pour diagnostic
diagnostic_samples = random.sample(results, min(10, len(results)))

for i, sample in enumerate(diagnostic_samples):
    print(f"\n{'='*60}")
    print(f"[{i+1}] Classe réelle: {sample['actual']}")
    print(f"    Classe prédite: {sample['predicted']}")
    print(f"    Correct: {'✅' if sample['correct'] else '❌'}")
    print(f"{'='*60}")
    print(f"RÉPONSE BRUTE:")
    print(f"{sample['response']}")
    if len(sample['response']) >= 200:
        print("...")

DIAGNOSTIC: RÉPONSES BRUTES DU MODÈLE

[1] Classe réelle: CGM
    Classe prédite: CGM
    Correct: ✅
RÉPONSE BRUTE:
disease: cassava green mottle (cgm)

cassava green mottle (cgm) is one of the most devastating cassava diseases, causing severe yield losses. it is distinguished by green mottling patterns on leaves. 
...

[2] Classe réelle: CGM
    Classe prédite: CGM
    Correct: ✅
RÉPONSE BRUTE:
disease: cassava green mottle (cgm)

cassava green mottle (cgm) is one of the most devastating cassava diseases, causing severe yield losses. it is distinguished by green mottling patterns on leaves. 
...

[3] Classe réelle: CMD
    Classe prédite: CMD
    Correct: ✅
RÉPONSE BRUTE:
disease: cassava mosaic disease (cmd)

cassava mosaic disease (cmd) is one of the most devastating cassava diseases, causing severe mosaic patterns, leaf distortion, and stunted growth. it is transmit
...

[4] Classe réelle: CBSD
    Classe prédite: CGM
    Correct: ❌
RÉPONSE BRUTE:
disease: cassava green mottle (cgm

In [15]:
# ============================================================
# CELL 14: ÉVALUATION VQA
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION V4 - VQA (Questions variées)")
print("="*60)

# Questions VQA variées (français)
vqa_questions = [
    "Quels sont les signes de maladie que vous observez ?",
    "Que dois-je faire pour traiter cette maladie ?",
    "Est-ce un problème de maladie ou d'insectes ?",
    "Qu'est-ce qui cause ces symptômes ?"
]

print("\n📊 Test VQA sur 10 samples...\n")
vqa_samples = random.sample(val_formatted, min(10, len(val_formatted)))

for i, sample in enumerate(vqa_samples):
    question = random.choice(vqa_questions)
    response = generate_response(sample['image_path'], question)
    
    print(f"{'='*60}")
    print(f"[{i+1}] Classe: {sample['class_name']}")
    print(f"    Question: {question}")
    print(f"    Réponse: {response[:300]}")
    if len(response) > 300:
        print("...")

print("\n" + "="*60)


ÉVALUATION V4 - VQA (Questions variées)

📊 Test VQA sur 10 samples...

[1] Classe: CGM
    Question: Qu'est-ce qui cause ces symptômes ?
    Réponse: la cause de ces problèmes est la cbsd, transmitte par les fourmis green cassava beetles. cependant, sans piqûres, il est difficile de déterminer le stade. les pois blanchis sont un signe distinctif. mesures à prendre : récolter quelques plants avant récolte principale. boutures saines certifiées (te
...
[2] Classe: CMD
    Question: Quels sont les signes de maladie que vous observez ?
    Réponse: cette image montre : motifs mosaïque jaune-vert sur les feuilles, déformation possible des folioles, motifs scellés (chambres blanches). ces signes sont typiques de la mosaïque du manioc.
[3] Classe: CBB
    Question: Est-ce un problème de maladie ou d'insectes ?
    Réponse: il s'agit de **bactériose du manioc** (cbb), causée par xanthomonas phaseoli pv. manihotis. les symptômes distinctifs sont : flétrissement foliaire, halo chlorotique jaune

In [16]:
# ============================================================
# CELL 15: SAUVEGARDE
# ============================================================

print("\n" + "="*60)
print("SAUVEGARDE MODÈLE V4")
print("="*60)

final_model_path = CHECKPOINT_DIR / "final_model"

# Sauvegarder le modèle avec les deux adapters
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
processor.save_pretrained(final_model_path)

# Sauvegarder configuration et résultats
config = {
    "stage": "2_v4_olora",
    "base_checkpoint": str(STAGE1_CHECKPOINT),
    "stage1_accuracy": STAGE1_RESULTS['global'],
    "v4_accuracy": global_accuracy,
    "training_config": TRAINING_CONFIG,
    "new_lora_config": NEW_LORA_CONFIG,
    "data_mix_config": DATA_MIX_CONFIG,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
    "train_samples": len(train_dataset),
    "class_metrics": class_metrics,
}

with open(CHECKPOINT_DIR / "training_config.json", 'w') as f:
    json.dump(config, f, indent=2)

eval_results = {
    "stage": "2_v4_olora",
    "total_samples": len(results),
    "global_accuracy_v4": global_accuracy,
    "global_accuracy_stage1": STAGE1_RESULTS['global'],
    "global_accuracy_v3": 1.0,
    "class_metrics_v4": class_metrics,
    "class_metrics_stage1": STAGE1_RESULTS,
    "improvement_vs_v3": global_accuracy - 1.0,
    "retention_vs_stage1": global_accuracy - STAGE1_RESULTS['global'],
}

with open(CHECKPOINT_DIR / "eval_results.json", 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Modèle sauvegardé: {final_model_path}")
print(f"✅ Configuration: {CHECKPOINT_DIR / 'training_config.json'}")
print(f"✅ Résultats: {CHECKPOINT_DIR / 'eval_results.json'}")

print("\n" + "="*60)
print("CONCLUSION V4")
print("="*60)

retention = global_accuracy - STAGE1_RESULTS['global']
improvement_v3 = global_accuracy - 1.0

if retention >= -5:
    print(f"\n🎉 Rétention excellente! ({retention:+.1f}% vs Stage 1)")
elif retention >= -15:
    print(f"\n✅ Rétention acceptable ({retention:+.1f}% vs Stage 1)")
elif retention >= -25:
    print(f"\n⚠️ Rétention modérée ({retention:+.1f}% vs Stage 1)")
else:
    print(f"\n❌ Rétention insuffisante ({retention:+.1f}% vs Stage 1)")

print(f"📈 Amélioration vs V3: +{improvement_v3:.1f}%")

print(f"\n📊 Résumé final:")
print(f"  - Stage 1: {STAGE1_RESULTS['global']:.1f}%")
print(f"  - V3 (échec): 1.0%")
print(f"  - V4 (O-LoRA): {global_accuracy:.1f}%")

print(f"\nCheckpoint: {final_model_path}")


SAUVEGARDE MODÈLE V4

✅ Modèle sauvegardé: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v4_olora/final_model
✅ Configuration: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v4_olora/training_config.json
✅ Résultats: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v4_olora/eval_results.json

CONCLUSION V4

❌ Rétention insuffisante (-42.0% vs Stage 1)
📈 Amélioration vs V3: +37.5%

📊 Résumé final:
  - Stage 1: 80.5%
  - V3 (échec): 1.0%
  - V4 (O-LoRA): 38.5%

Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v4_olora/final_model
